The Blood-brain barrier penetration (BBBP)
--------------------------------------------------------------

### All libraries we need

In [1]:
import warnings
warnings.filterwarnings('ignore')


import os
import os.path as osp
import shutil
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import copy
import time


import torch
import torch.nn as nn
from torch.optim import Adam
from  utils import *

#********************************************************#
'''
load_dataset contain lots of functions for loading several datasets and 
also there is a function as name get_ dataloader for generating a
dictionary of training, validation, and testing dataLoader.
'''
from load_dataset import get_dataset, get_dataloader

#********************************************************#
'''
As we need several arguments for training process, we store all argument in configure file. 
For using this file, you need the library'Typed Argument Parser (Tap). So you need 'pip install typed-argument-parser'. 
'''
from Configures import data_args, train_args, model_args

### start loading data

In [2]:
print(data_args.dataset_name)
print(data_args.dataset_dir)

bbbp
/datasets


In [3]:
dataset = get_dataset(data_args.dataset_dir, data_args.dataset_name)
input_dim = dataset.num_node_features
output_dim = int(dataset.num_classes)


print(input_dim)
print(output_dim)

9
2


### Data Analysis

In [4]:
avg_nodes = 0.0
avg_edge_index = 0.0
for i in range(len(dataset)):
    avg_nodes += dataset[i].x.shape[0]
    avg_edge_index += dataset[i].edge_index.shape[1]
avg_nodes /= len(dataset)
avg_edge_index /= len(dataset)
print(f"graphs {len(dataset)}, avg_nodes{avg_nodes :.4f}, avg_edge_index_{avg_edge_index/2 :.4f}")

best_acc = 0.0
data_size = len(dataset)
print(f'The total num of dataset is {data_size}')



graphs 2050, avg_nodes23.9356, avg_edge_index_25.8151
The total num of dataset is 2050


In [5]:


# Read the CSV file
df = pd.read_csv('datasets/bbbp/raw/BBBP.csv')

# Print the shape of the dataset
print("The shape of the dataset is:", df.shape)

# Print the columns of the dataset
print("The columns of the dataset are:", df.columns)

# Print the summary statistics of the dataset
print("The summary statistics of the dataset are:")
print(df.describe())

# Print some sample rows of the dataset
print("Some sample rows of the dataset are:")
df.head(5)

The shape of the dataset is: (2050, 4)
The columns of the dataset are: Index(['num', 'name', 'p_np', 'smiles'], dtype='object')
The summary statistics of the dataset are:
               num         p_np
count  2050.000000  2050.000000
mean   1027.376098     0.764390
std     592.836849     0.424483
min       1.000000     0.000000
25%     514.250000     1.000000
50%    1026.500000     1.000000
75%    1540.750000     1.000000
max    2053.000000     1.000000
Some sample rows of the dataset are:


,num,name,p_np,smiles
0,1,Propanolol,1,[Cl].CC(C)NCC(O)COc1cccc2ccccc12
1,2,Terbutylchlorambucil,1,C(=O)(OC(C)(C)C)CCCc1ccc(cc1)N(CCCl)CCCl
2,3,40730,1,c12c3c(N4CCN(C)CC4)c(F)cc1c(c(C(O)=O)cn2C(C)CO...
3,4,24,1,C1CCN(CC1)Cc1cccc(c1)OCCCNC(=O)C
4,5,cloxacillin,1,Cc1onc(c2ccccc2Cl)c1C(=O)N[C@H]3[C@H]4SC(C)(C)...


### Visualizing

In [12]:
smiles_list = df["smiles"][10:20].tolist()
name_list = df["name"][10:20].tolist()
label = df["p_np"][10:20].tolist()
new_label=[]
for i in range(10): 
        if label[i]==1:
                new_label.append( 'blood-brain barrier' )
        else :   
                new_label.append('non-blood-brain barrier')

                # Convert each sublist in new_label to a string
new_label_str = [' - '.join(sublist) for sublist in new_label]
#new_label=[for i in len(label): 'blood-brain barrier' if label[i]==1 else 'non-blood-brain barrier']

In [13]:
new_label

['non-blood-brain barrier',
 'blood-brain barrier',
 'blood-brain barrier',
 'blood-brain barrier',
 'blood-brain barrier',
 'blood-brain barrier',
 'blood-brain barrier',
 'blood-brain barrier',
 'blood-brain barrier',
 'blood-brain barrier']

In [59]:
smiles_list = df["smiles"][10:20].tolist()
name_list = df["name"][10:20].tolist()
label = df["p_np"][10:20].tolist()
new_label=[]
for i in range(len(label)): 
        if label[i]==1:
                new_label.append( name_list[i]+'- blood-brain barrier')
        else :   
                new_label.append(name_list[i]+'- non-blood-brain barrier')

In [15]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw

# Read the CSV file
df = pd.read_csv('datasets/bbbp/raw/BBBP.csv')

# Extract the SMILES strings and names of the first 6 compounds
smiles_list = df["smiles"][10:16].tolist()
name_list = df["name"][10:16].tolist()
label = df["p_np"][10:16].tolist()
new_label=[]
for i in range(len(label)): 
        if label[i]==1:
                new_label.append( name_list[i]+'- blood-brain barrier')
        else :   
                new_label.append(name_list[i]+'- non-blood-brain barrier')

	
plt.rcParams['figure.figsize'] = [20, 5] 
plt.rcParams.update({'font.size': 12})
# Convert the SMILES strings into RDKit molecule objects
mol_list = [Chem.MolFromSmiles(smiles) for smiles in smiles_list]

# Create a grid image with 2 rows and 3 columns and put the names as legends
img = Draw.MolsToGridImage(mol_list, molsPerRow=3,subImgSize=(300, 300), legends=new_label)

# img is an IPython.display.Image object, we can get raw PNG data from it
png = img.data

# Write raw PNG data to file
with open("Images/BBBP-sample.png", "wb") as f:
    f.write(png)

In [16]:
img = Draw.MolsToGridImage(mol_list, molsPerRow=5, subImgSize=(300, 300), legends=new_label, useSVG=False)

# img is an IPython.display.Image object, we can get raw PNG data from it
png = img.data

# Write raw PNG data to file
with open("output.png", "wb") as f:
    f.write(png)

### Preprocessing and cleaning dataset

In [4]:
#cleaned_dataset = [graph for graph in dataset if graph.edge_index.numpy()!=[]]
cleaned_dataset = [graph for graph in dataset if graph.edge_index.numpy().size> 0]
cleaned_dataset_len=len(cleaned_dataset)
print(f'The number of graphs after cleaning dataset is: {cleaned_dataset_len}')

The number of graphs after cleaning dataset is: 2039


In [5]:
dataloader=get_dataloader(cleaned_dataset, batch_size=train_args.batch_size, random_split_flag=True, data_split_ratio=[0.8, 0.1, 0.1], seed=2)

### Traninig Process

In [7]:
from GCN import GCNNet

def get_model(input_dim, output_dim, model_args):
    if model_args.model_name.lower() == 'gcn':
        return GCNNet(input_dim, output_dim, model_args)
    elif model_args.model_name.lower() == 'gat':
        return GATNet(input_dim, output_dim, model_args)
    elif model_args.model_name.lower() == 'gin':
        return GINNet(input_dim, output_dim, model_args)
    else:
        raise NotImplementedError
        


class GnnBase(nn.Module):
    def __init__(self):
        super(GnnBase, self).__init__()

    def forward(self, data):
        data = data.to(self.device)
        logits, prob, emb = self.model(data)
        return logits, prob, emb

    def update_state_dict(self, state_dict):
        original_state_dict = self.state_dict()
        loaded_state_dict = dict()
        for k, v in state_dict.items():
            if k in original_state_dict.keys():
                loaded_state_dict[k] = v
        self.load_state_dict(loaded_state_dict)

    def to_device(self):
        self.to(self.device)

    def save_state_dict(self):
        pass


class GnnNets(GnnBase):
    def __init__(self, input_dim, output_dim, model_args):
        super(GnnNets, self).__init__()
        self.model = get_model(input_dim, output_dim, model_args)
        self.device = model_args.device

    def forward(self, data):
        data = data.to(self.device)
        logits, prob, emb = self.model(data)
        return logits, prob, emb



In [8]:
def evaluate_GC(eval_dataloader,model, criterion):
    acc = []
    loss_list = []
    model.eval()
    with torch.no_grad():
        for batch in eval_dataloader:
            logits, probs, _ = model(batch)
            loss = criterion(logits, batch.y)

            ## record
            _, prediction = torch.max(logits, -1)
            loss_list.append(loss.item())
            acc.append(prediction.eq(batch.y).cpu().numpy())

        eval_state = {'loss': np.average(loss_list),
                      'acc': np.concatenate(acc, axis=0).mean()}

    return eval_state


def test_GC(test_dataloader,model, criterion):
    acc = []
    loss_list = []
    pred_probs = []
    predictions = []
    model.eval()
    with torch.no_grad():
        for batch in test_dataloader:
            logits, probs, _ = model(batch)
            loss = criterion(logits, batch.y)

            # record
            _, prediction = torch.max(logits, -1)
            loss_list.append(loss.item())
            acc.append(prediction.eq(batch.y).cpu().numpy())
            predictions.append(prediction)
            pred_probs.append(probs)

    test_state = {'loss': np.average(loss_list),
                  'acc': np.average(np.concatenate(acc, axis=0).mean())}

    pred_probs = torch.cat(pred_probs, dim=0).cpu().detach().numpy()
    predictions = torch.cat(predictions, dim=0).cpu().detach().numpy()
    return test_state, pred_probs, predictions

def save_best(ckpt_dir, epoch, model, model_name, eval_acc, is_best):
    print('saving....')
    model.to('cpu')
    state = {
        'net': model.state_dict(),
        'epoch': epoch,
        'acc': eval_acc
    }
    pth_name = f"{model_name}_latest.pth"
    best_pth_name = f'{model_name}_best.pth'
    ckpt_path = os.path.join(ckpt_dir, pth_name)
    torch.save(state, ckpt_path)
    if is_best:
        shutil.copy(ckpt_path, os.path.join(ckpt_dir, best_pth_name))
    model.to_device()

### save path for model

In [9]:

if not os.path.isdir('checkpoint'):
    os.mkdir('checkpoint')
if not os.path.isdir(os.path.join('checkpoint', data_args.dataset_name)):
    os.mkdir(os.path.join('checkpoint', f"{data_args.dataset_name}"))
ckpt_dir = f"./checkpoint/{data_args.dataset_name}/"



In [18]:
def train(l2):
    logits, probs, _ = model(batch)
    loss = criterion(logits, batch.y)
    l2_reg = torch.tensor(0.)
    for param in model.parameters():
        l2_reg += torch.norm(param)

    # Combine the loss function with L2 regularization
    loss += (l2 * l2_reg)
    
    # optimization
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_value_(model.parameters(), clip_value=2.0)
    optimizer.step()
    
    ## record
    _, prediction = torch.max(logits, -1)
    loss_list.append(loss.item())
    acc.append(prediction.eq(batch.y).cpu().numpy())
    
  
    return acc

In [10]:
def train():
    logits, probs, _ = model(batch)
    loss = criterion(logits, batch.y)
    
    # optimization
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_value_(model.parameters(), clip_value=2.0)
    optimizer.step()
    
    ## record
    _, prediction = torch.max(logits, -1)
    loss_list.append(loss.item())
    acc.append(prediction.eq(batch.y).cpu().numpy())
    
  
    return acc

#### Training

In [14]:

train_args.weight_decay=0.0
model = GnnNets(input_dim, output_dim, model_args)
model.to_device()
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=train_args.learning_rate, weight_decay=train_args.weight_decay)

In [15]:
best_acc=0
early_stop_count = 0
#l2_lambda=0.9
for epoch in range(10):
  
    acc=[]
    loss_list = []
    model.train()
    for batch in dataloader['train']:
        acc=train(l2)
    
    # report train msg
    print(f"Train Epoch:{epoch}  |Loss: {np.average(loss_list):.3f} | "
          f"Acc: {np.concatenate(acc, axis=0).mean():.3f}")
    
    # report eval msg
    eval_state = evaluate_GC(dataloader['eval'], model, criterion)
    print(f"Eval Epoch: {epoch} | Loss: {eval_state['loss']:.3f} | Acc: {eval_state['acc']:.3f}")
    
    # only save the best model
    is_best = (eval_state['acc'] > best_acc)

    if eval_state['acc'] > best_acc:
        early_stop_count = 0
    else:
        early_stop_count += 1

    if early_stop_count > train_args.early_stopping:
        break

    if is_best:
        best_acc = eval_state['acc']
        early_stop_count = 0
    if is_best or epoch % train_args.save_epoch == 0:
        save_best(ckpt_dir, epoch, model, model_args.model_name, eval_state['acc'], is_best)

print(f"The best validation accuracy is {best_acc}.")
  


Train Epoch:0  |Loss: 0.572 | Acc: 0.773
Eval Epoch: 0 | Loss: 0.543 | Acc: 0.754
saving....
Train Epoch:1  |Loss: 0.547 | Acc: 0.773
Eval Epoch: 1 | Loss: 0.534 | Acc: 0.754
Train Epoch:2  |Loss: 0.524 | Acc: 0.773
Eval Epoch: 2 | Loss: 0.521 | Acc: 0.754
Train Epoch:3  |Loss: 0.516 | Acc: 0.772
Eval Epoch: 3 | Loss: 0.492 | Acc: 0.754
Train Epoch:4  |Loss: 0.500 | Acc: 0.774
Eval Epoch: 4 | Loss: 0.472 | Acc: 0.798
saving....
Train Epoch:5  |Loss: 0.471 | Acc: 0.779
Eval Epoch: 5 | Loss: 0.487 | Acc: 0.788
Train Epoch:6  |Loss: 0.491 | Acc: 0.776
Eval Epoch: 6 | Loss: 0.493 | Acc: 0.773
Train Epoch:7  |Loss: 0.459 | Acc: 0.793
Eval Epoch: 7 | Loss: 0.493 | Acc: 0.808
saving....
Train Epoch:8  |Loss: 0.471 | Acc: 0.797
Eval Epoch: 8 | Loss: 0.456 | Acc: 0.818
saving....
Train Epoch:9  |Loss: 0.467 | Acc: 0.801
Eval Epoch: 9 | Loss: 0.474 | Acc: 0.813
The best validation accuracy is 0.8177339901477833.


In [16]:
 print(f"Acc: {np.concatenate(acc, axis=0).mean():.3f}")

Acc: 0.801


### Evaluation 

In [16]:
print(f"The best validation accuracy is {best_acc}.")
# report test msg
checkpoint = torch.load(os.path.join(ckpt_dir, f'{model_args.model_name}_best.pth'))
model.update_state_dict(checkpoint['net'])
test_state, _, _ = test_GC(dataloader['test'], model, criterion)
print(f"Test: | Loss: {test_state['loss']:.3f} | Acc: {test_state['acc']:.3f}")

The best validation accuracy is 0.7536945812807881.
Test: | Loss: 0.655 | Acc: 0.717


## Manual Measurement

In [19]:
import statistics as stat
#train_args.weight_decay=0.0001
l2_lambda = 0.0001
Eva_final=dict()
num_epoch=100
Accuracy=[]
T_inference=[]
Num_parm=[]
Model_size=[]

In [20]:
for i in range(5):

        Eva=dict()
        model = GnnNets(input_dim, output_dim, model_args)
        model.to_device()
        criterion = nn.CrossEntropyLoss()
        optimizer = Adam(model.parameters(), lr=train_args.learning_rate, weight_decay=train_args.weight_decay)

        for epoch in range(num_epoch):  
            acc=[]
            loss_list = []
            model.train()
            for batch in dataloader['train']:
                acc=train(l2_lambda)       

            eval_state = evaluate_GC(dataloader['eval'], model, criterion)
            accuracy=eval_state['acc']

            if epoch % 20 == 0:   
                # report train msg
                print(f"Train Epoch:{epoch}  |Loss: {np.average(loss_list):.3f} | "
                      f"Acc: {np.concatenate(acc, axis=0).mean():.3f}")
                print(f"Eval Epoch: {epoch} | Loss: {eval_state['loss']:.3f} | Acc: {accuracy:.3f}")



        best_checkpoint = dict()
        best_checkpoint['state_dict'] = copy.deepcopy(model.state_dict())
        model.load_state_dict(best_checkpoint['state_dict'])
        recover_model = lambda: model.load_state_dict(best_checkpoint['state_dict'])

        t0=time.time()
        test_state, _, _ = test_GC(dataloader['test'], model, criterion)
        test_acc= test_state['acc']
        t1=time.time()
        t_inference=t1 - t0
        ###
        model_size = get_model_size(model,count_nonzero_only=True)
        num_parm=get_num_parameters(model, count_nonzero_only=True)
        # Print all measurment
        print(f"Our model with {l2_lambda} regularization has accuracy on test set={test_acc:.2f}%")
        print(f"Our model with {l2_lambda} regularization has size={model_size/MiB:.2f} MiB")
        print(f"The time inference of our model with {l2_lambda} regularization is ={t_inference}") 
        print(f"The number of parametrs of our model with {l2_lambda} regularization is:{num_parm}")



        #Update my Eva dictionary
        Eva.update({'Model accuracy': test_acc,
                    'time inference': t_inference,
                    'number parmameters of model': num_parm,
                    'size of model': model_size})


        Accuracy.append(Eva['Model accuracy'])
        T_inference.append(Eva['time inference'])
        Num_parm.append(int(Eva['number parmameters of model']))
        Model_size.append(int(Eva['size of model']))

Train Epoch:0  |Loss: 0.560 | Acc: 0.773
Eval Epoch: 0 | Loss: 0.553 | Acc: 0.754
Train Epoch:20  |Loss: 0.433 | Acc: 0.819
Eval Epoch: 20 | Loss: 0.430 | Acc: 0.833
Train Epoch:40  |Loss: 0.385 | Acc: 0.845
Eval Epoch: 40 | Loss: 0.404 | Acc: 0.857
Train Epoch:60  |Loss: 0.347 | Acc: 0.860
Eval Epoch: 60 | Loss: 0.333 | Acc: 0.877
Train Epoch:80  |Loss: 0.328 | Acc: 0.866
Eval Epoch: 80 | Loss: 0.310 | Acc: 0.877
Our model with 0.0001 regularization has accuracy on test set=0.80%
Our model with 0.0001 regularization has size=0.13 MiB
The time inference of our model with 0.0001 regularization is =0.13802671432495117
The number of parametrs of our model with 0.0001 regularization is:34562
Train Epoch:0  |Loss: 0.562 | Acc: 0.764
Eval Epoch: 0 | Loss: 0.553 | Acc: 0.754
Train Epoch:20  |Loss: 0.427 | Acc: 0.831
Eval Epoch: 20 | Loss: 0.432 | Acc: 0.828
Train Epoch:40  |Loss: 0.383 | Acc: 0.844
Eval Epoch: 40 | Loss: 0.381 | Acc: 0.842
Train Epoch:60  |Loss: 0.343 | Acc: 0.861
Eval Epoch:

In [21]:
Eva_final=dict()
std=dict()
print(100 * "=")

model_accuracy_mean = stat.mean(Accuracy)
model_accuracy_std =  stat.stdev(Accuracy)
Eva_final.update({'model accuracy':float(format(model_accuracy_mean, '.4f'))})
std.update({'Std of base model accuracy':float(format(model_accuracy_std, '.4f'))})

desc_model_accuracy = "{:.3f} ± {:.3f}".format(model_accuracy_mean,model_accuracy_std)
print(f"The mean of model accuracy:{desc_model_accuracy}")


print(100 * "=")
####Time inference    
t_model_mean =stat.mean(T_inference)
t_model_std =stat.stdev(T_inference)
Eva_final.update({'time inference of model':float(format(t_model_mean, '.6f'))})
std.update({'Std of time inference of model':float(format(t_model_std, '.6f'))})

desc_t_model = "{:.3f} ± {:.3f}".format(t_model_mean,t_model_std)
print(f"The mean of inference time is :{desc_t_model} ")

print(100 * "=")
####Number of Parameters   
num_parm_model_mean = stat.mean(Num_parm)
Eva_final.update({'number parmameters of model':num_parm_model_mean})
print(f"The number of parameters is :{num_parm_model_mean} ")

print(100 * "=")
####Model Size 
model_size_mean = stat.mean(Model_size)
Eva_final.update({'base_model_size':model_size_mean})
print(f"The model size is :{model_size_mean} ")



The mean of model accuracy:0.826 ± 0.018
The mean of inference time is :0.139 ± 0.012 
The number of parameters is :34562 
The model size is :1105984 


In [22]:
BBBP_0001=Eva_final
print(BBBP_0001)

{'model accuracy': 0.8263, 'time inference of model': 0.138546, 'number parmameters of model': 34562, 'base_model_size': 1105984}


In [23]:
BBBP_std_0001=std
print(BBBP_std_0001)

{'Std of base model accuracy': 0.0184, 'Std of time inference of model': 0.012372}


In [ ]:
BBBP_std_0001={'Std of base model accuracy': 0.0184, 'Std of time inference of model': 0.012372}
